In [0]:
%pip install faker

In [0]:
dbutils.library.restartPython()

In [0]:
import random
import uuid
from datetime import datetime, timedelta
 
import pandas as pd
from faker import Faker
 
fake = Faker("en_GB")   
Faker.seed(42)
random.seed(42)
 
print("Faker version:", Faker.VERSION if hasattr(Faker, "VERSION") else "installed")
print("Seed set. Data will be identical on every run.")

In [0]:
CONFIG = {
    "n_customers":    50_000,
    "n_accounts":     55_000,   # slightly more than customers — some have >1 account
    "n_transactions": 200_000,  # high volume to make DQ rules statistically meaningful
 
    # Dirt rates — what % of records get each type of intentional error
    "null_rate":      0.03,     # 3% of optional fields go NULL
    "duplicate_rate": 0.01,     # 1% of rows are duplicated
    "future_date_pct":0.005,    # 0.5% of transaction dates are in the future
    "invalid_email_pct": 0.02,  # 2% of emails are malformed
 
    # Workspace path — raw CSVs land here
    "volume_path": "/Workspace/Users/kamranhabib1212@gmail.com/datasentry/raw_data",
}
 
print("Config loaded:")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

In [0]:
def inject_nulls(series: pd.Series, rate: float) -> pd.Series:
    """Replace `rate` fraction of values in a Series with None."""
    mask = pd.Series(
        [random.random() < rate for _ in range(len(series))],
        index=series.index
    )
    return series.where(~mask, other=None)

In [0]:
print("Generating customers...")
 
n = CONFIG["n_customers"]
customer_ids = [str(uuid.uuid4()) for _ in range(n)]
 
# Build base records
customers = pd.DataFrame({
    "customer_id":   customer_ids,
    "full_name":     [fake.name() for _ in range(n)],
    "email":         [fake.email() for _ in range(n)],
    "phone":         [fake.phone_number() for _ in range(n)],
    "date_of_birth": [
        fake.date_of_birth(minimum_age=18, maximum_age=80).isoformat()
        for _ in range(n)
    ],
    "country":       [
        random.choice(["IE"] * 85 + ["GB", "US", "DE", "FR", "IN"] * 3)
        for _ in range(n)
    ],
    "created_at":    [
        fake.date_time_between(start_date="-5y", end_date="now").isoformat()
        for _ in range(n)
    ],
})

# --- Inject malformed emails (missing @ symbol) ---
bad_email_idx = customers.sample(frac=CONFIG["invalid_email_pct"]).index
customers.loc[bad_email_idx, "email"] = customers.loc[bad_email_idx, "email"].str.replace("@", "", regex=False)
 
# --- Inject nulls into optional fields ---
customers["phone"]         = inject_nulls(customers["phone"],         CONFIG["null_rate"])
customers["date_of_birth"] = inject_nulls(customers["date_of_birth"], CONFIG["null_rate"])
 
# --- Inject duplicate rows (1% duplication) ---
n_dupes = int(n * CONFIG["duplicate_rate"])
dupes = customers.sample(n=n_dupes, replace=True)
customers = pd.concat([customers, dupes], ignore_index=True)
 
print(f"  Rows (incl. dupes): {len(customers):,}")
print(f"  Null phones:        {customers['phone'].isna().sum():,}")
print(f"  Malformed emails:   {customers['email'].str.contains('@').eq(False).sum():,}")
customers.head(3)

In [0]:
print("Generating accounts...")
 
n = CONFIG["n_accounts"]
 
# Most accounts belong to real customers; ~2% get a random (non-existent) customer_id
def pick_customer_id(valid_ids, orphan_rate=0.02):
    if random.random() < orphan_rate:
        return str(uuid.uuid4())   # orphan — FK violation
    return random.choice(valid_ids)
 
account_customer_ids = [pick_customer_id(customer_ids) for _ in range(n)]
 
account_types = [random.choice(["CURRENT", "SAVINGS", "LOAN"]) for _ in range(n)]
 
# CURRENT accounts can go negative (overdraft); others are non-negative
balances = []
for atype in account_types:
    if atype == "CURRENT":
        balances.append(round(random.uniform(-500, 50_000), 2))
    elif atype == "SAVINGS":
        balances.append(round(random.uniform(0, 100_000), 2))
    else:  # LOAN — outstanding balance
        balances.append(round(random.uniform(1_000, 50_000), 2))
 
accounts = pd.DataFrame({
    "account_id":   [str(uuid.uuid4()) for _ in range(n)],
    "customer_id":  account_customer_ids,
    "account_type": account_types,
    "status":       [random.choice(["ACTIVE", "ACTIVE", "ACTIVE", "INACTIVE", "CLOSED"]) for _ in range(n)],
    "balance":      balances,
    "currency":     [random.choice(["EUR"] * 80 + ["GBP"] * 15 + ["USD"] * 5) for _ in range(n)],
    "opened_date":  [
        fake.date_between(start_date="-10y", end_date="today").isoformat()
        for _ in range(n)
    ],
})
 
# Inject nulls into balance
accounts["balance"] = inject_nulls(accounts["balance"], CONFIG["null_rate"])
 
print(f"  Rows:           {len(accounts):,}")
print(f"  Null balances:  {accounts['balance'].isna().sum():,}")
print(f"  Orphaned FKs:   ~{int(n * 0.02):,} (approx)")
accounts.head(3)

In [0]:
print("Generating transactions...")
 
n = CONFIG["n_transactions"]
account_id_list = accounts["account_id"].tolist()
 
# Some transactions reference non-existent accounts (2% orphan rate)
txn_account_ids = [
    pick_customer_id(account_id_list, orphan_rate=0.02)
    for _ in range(n)
]
 
# Generate dates — mostly historical, 0.5% future
today = datetime.today()
def gen_txn_date(future_rate=CONFIG["future_date_pct"]):
    if random.random() < future_rate:
        # Future date — up to 30 days ahead
        return (today + timedelta(days=random.randint(1, 30))).isoformat()
    return fake.date_time_between(start_date="-3y", end_date="now").isoformat()
 
txn_dates = [gen_txn_date() for _ in range(n)]
 
# Amounts — mostly valid, 1% zero
def gen_amount():
    if random.random() < 0.01:
        return 0.00   # invalid zero amount
    return round(random.uniform(0.01, 10_000), 2)
 
amounts = [gen_amount() for _ in range(n)]
 
transactions = pd.DataFrame({
    "transaction_id":   [str(uuid.uuid4()) for _ in range(n)],
    "account_id":       txn_account_ids,
    "transaction_type": [random.choice(["DEBIT", "CREDIT", "TRANSFER"]) for _ in range(n)],
    "amount":           amounts,
    "currency":         [random.choice(["EUR"] * 78 + ["GBP"] * 15 + ["USD"] * 5 + ["XYZ"] * 2) for _ in range(n)],
    "transaction_date": txn_dates,
    "merchant":         [fake.company() for _ in range(n)],
    "status":           [
        random.choice(["COMPLETED"] * 80 + ["PENDING"] * 10 + ["FAILED"] * 7 + ["REVERSED"] * 3)
        for _ in range(n)
    ],
})
 
# Inject nulls into merchant
transactions["merchant"] = inject_nulls(transactions["merchant"], CONFIG["null_rate"])
 
# Inject duplicate rows
n_dupes = int(n * CONFIG["duplicate_rate"])
dupes = transactions.sample(n=n_dupes, replace=True)
transactions = pd.concat([transactions, dupes], ignore_index=True)
 
print(f"  Rows (incl. dupes):     {len(transactions):,}")
print(f"  Future dates:           {pd.to_datetime(transactions['transaction_date']) > today}.sum()")
print(f"  Zero-amount rows:       {(transactions['amount'] == 0).sum():,}")
print(f"  Null merchants:         {transactions['merchant'].isna().sum():,}")
transactions.head(3)

In [0]:
import os
 
volume_path = CONFIG["volume_path"]
 
# Create directory inside the Volume (safe if already exists)
dbutils.fs.mkdirs(volume_path)
 
# Write CSVs
customers.to_csv(f"{volume_path}/customers.csv",    index=False)
accounts.to_csv(f"{volume_path}/accounts.csv",      index=False)
transactions.to_csv(f"{volume_path}/transactions.csv", index=False)
 
print("Files written to Volume:")
for fname in ["customers.csv", "accounts.csv", "transactions.csv"]:
    fpath = f"{volume_path}/{fname}"
    size_mb = os.path.getsize(fpath) / (1024 * 1024)
    rows = pd.read_csv(fpath).shape[0]
    print(f"  {fname}: {rows:,} rows | {size_mb:.2f} MB")


In [0]:
# Create schema and tables from pandas DataFrames

from pyspark.sql import SparkSession

# Set catalog and create schema
spark.sql("USE CATALOG workspace")
spark.sql("""
    CREATE SCHEMA IF NOT EXISTS banking_datasentry
    COMMENT 'Banking data with intentional quality issues for DataSentry testing'
""")
spark.sql("USE SCHEMA banking_datasentry")

print("Creating customers table...")
customers_spark = spark.createDataFrame(customers)
customers_spark.write.mode("overwrite").saveAsTable("customers")
spark.sql("COMMENT ON TABLE customers IS 'Customer master data with intentional duplicates and malformed emails'")

print("Creating accounts table...")
accounts_spark = spark.createDataFrame(accounts)
accounts_spark.write.mode("overwrite").saveAsTable("accounts")
spark.sql("COMMENT ON TABLE accounts IS 'Account data with null balances and orphaned foreign keys'")

print("Creating transactions table...")
transactions_spark = spark.createDataFrame(transactions)
transactions_spark.write.mode("overwrite").saveAsTable("transactions")
spark.sql("COMMENT ON TABLE transactions IS 'Transaction data with future dates, zero amounts, and duplicates'")

# Verify table creation
print("\nTable creation complete! Verifying row counts...")
result = spark.sql("""
    SELECT 'customers' AS table_name, COUNT(*) AS row_count FROM customers
    UNION ALL
    SELECT 'accounts', COUNT(*) FROM accounts
    UNION ALL
    SELECT 'transactions', COUNT(*) FROM transactions
""")

display(result)